<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/13-performance-tuning/03-table-statistics-cbo.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 13.3 — Table statistics: give the optimizer real numbers

Compare `explain(mode="cost")` before and after `ANALYZE TABLE`,
watch `rowCount` appear, then demonstrate the stale-stats trap: grow
a table without re-analyzing and show the optimizer still trusting
the old number.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("13.3-table-statistics-cbo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [ ]:
import shutil
spark.sql("DROP TABLE IF EXISTS orders_stats")
orders = (spark.range(500_000)
          .withColumn("customer_id", (col("id") % 2000).cast("int"))
          .withColumn("amount", col("id") % 500))
orders.write.mode("overwrite").saveAsTable("orders_stats")

## Before `ANALYZE`: cost-mode plan has size but no row count

In [ ]:
print("BEFORE ANALYZE:")
spark.table("orders_stats").explain(mode="cost")
# look for Statistics(sizeInBytes=...) with NO rowCount -- Spark has size for free,
# not row count (11.1's distinction, cashed in)

## Run `ANALYZE TABLE` — row count appears

In [ ]:
spark.sql("ANALYZE TABLE orders_stats COMPUTE STATISTICS")
spark.sql("ANALYZE TABLE orders_stats COMPUTE STATISTICS FOR COLUMNS customer_id, amount")

print("AFTER ANALYZE:")
spark.table("orders_stats").explain(mode="cost")
# now Statistics(...) should include rowCount

spark.sql("DESCRIBE EXTENDED orders_stats").show(50, truncate=False)

## The stale-stats trap: grow the table, don't re-analyze

The optimizer keeps trusting the OLD row count until you tell it
otherwise — silently, no error.

In [ ]:
more_orders = (spark.range(500_000, 5_500_000)   # 10x more rows
               .withColumn("customer_id", (col("id") % 2000).cast("int"))
               .withColumn("amount", col("id") % 500))
more_orders.write.mode("append").saveAsTable("orders_stats")

print(f"actual row count now: {spark.table('orders_stats').count():,}")
print("\nbut the STORED stats still say (stale, pre-append):")
spark.table("orders_stats").explain(mode="cost")
# rowCount in the plan is the OLD value -- the table is 10x bigger and the
# optimizer doesn't know it yet

## Fix: re-analyze, watch the number correct itself

In [ ]:
spark.sql("ANALYZE TABLE orders_stats COMPUTE STATISTICS")
print("AFTER re-ANALYZE (post-growth):")
spark.table("orders_stats").explain(mode="cost")

## CBO toggle: does it change a 3-way join's plan?

`spark.sql.cbo.enabled` unlocks cost-based join reordering. Not
guaranteed to visibly reorder on tiny demo data, but confirm the
config exists and inspect the plan either way.

In [ ]:
small_dim = spark.range(50).withColumnRenamed("id", "customer_id") \
                             .withColumn("segment", (col("customer_id") % 4).cast("int"))
small_dim.write.mode("overwrite").saveAsTable("segments_stats")
spark.sql("ANALYZE TABLE segments_stats COMPUTE STATISTICS FOR COLUMNS customer_id, segment")

three_way = spark.table("orders_stats").join(small_dim, "customer_id")

print("cbo.enabled =", spark.conf.get("spark.sql.cbo.enabled"))
three_way.explain(mode="cost")

spark.conf.set("spark.sql.cbo.enabled", "true")
print("\nwith CBO explicitly enabled:")
three_way.explain(mode="cost")
spark.conf.set("spark.sql.cbo.enabled", "false")

In [ ]:
spark.sql("DROP TABLE IF EXISTS orders_stats")
spark.sql("DROP TABLE IF EXISTS segments_stats")
spark.stop()
shutil.rmtree("spark-warehouse", ignore_errors=True)
shutil.rmtree("metastore_db", ignore_errors=True)

## Exercises

1. Re-run the before/after `ANALYZE` comparison but only run
   `COMPUTE STATISTICS` (table-level, no `FOR COLUMNS`). Does
   `rowCount` still appear? What's still missing that column-level
   stats would add?
2. After the append-without-reanalyze step, run an actual broadcast
   join decision test: join `orders_stats` against a table just over
   `autoBroadcastJoinThreshold`. Does the STALE size estimate cause a
   wrong broadcast decision? (Recall: byte size is free/always
   current — is rowCount staleness the same risk as size staleness?)
3. `DESCRIBE EXTENDED` prints a lot. Find the specific line(s) that
   store row count and size, and the column stats for `customer_id`
   after your `FOR COLUMNS` call.
4. Design an experiment where CBO's join reordering visibly changes
   the physical plan (hint: you'll likely need three tables of very
   different, ANALYZE'd sizes, with the "natural" join order being
   deliberately bad).
5. Sketch a scheduled-job addition (pseudocode, doesn't need to run)
   that re-runs `ANALYZE TABLE` after every load into `orders_stats`
   as part of the same pipeline — where in the pipeline does it need
   to sit relative to the write?

In [ ]:
# your exercise code here